# D4Explainer — Colab Setup
**Runtime:** GPU (T4 or better) — set via *Runtime > Change runtime type*

In [ ]:
# Cell 1: Upload and unzip code
from google.colab import files
import zipfile, os

print('Select D4Explainer.zip when the picker opens...')
uploaded = files.upload()  # select D4Explainer.zip

with zipfile.ZipFile('D4Explainer.zip', 'r') as z:
    z.extractall('/content/')

os.chdir('/content/D4Explainer')
print('CWD:', os.getcwd())

In [ ]:
# Cell 2: Upload and unzip datasets
# Upload one zip per dataset — repeat for each: MUTAG, BA3, BA_shapes, etc.
from google.colab import files
import zipfile, os

os.makedirs('/content/D4Explainer/data', exist_ok=True)

print('Select dataset zip(s) when the picker opens...')
uploaded = files.upload()  # can select multiple files

for fname in uploaded:
    if fname.endswith('.zip'):
        with zipfile.ZipFile(fname, 'r') as z:
            z.extractall('/content/D4Explainer/data/')
        print(f'Extracted {fname}')

print('\ndata/ contents:', os.listdir('/content/D4Explainer/data'))

In [ ]:
# Cell 3: Verify GPU
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
else:
    print('No GPU — check Runtime > Change runtime type > T4 GPU')

In [ ]:
# Cell 3b: Prevent TensorFlow GPU pre-allocation and verify clean GPU state
# TF is installed in Colab alongside PyTorch and grabs all GPU memory by default.
# Must run this before any training — if GPU memory looks wrong, restart the runtime first.
import os
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'

import torch
allocated = torch.cuda.memory_allocated() / 1024**3
reserved  = torch.cuda.memory_reserved()  / 1024**3
total     = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Total VRAM : {total:.1f} GiB')
print(f'Allocated  : {allocated:.2f} GiB  (should be ~0 on a fresh runtime)')
print(f'Reserved   : {reserved:.2f} GiB')
if allocated > 1.0:
    print('WARNING: >1 GiB already allocated — restart the runtime to clear leftover memory.')

In [ ]:
# Cell 4: Install dependencies
# Note: torch-scatter/sparse/cluster/spline-conv are pre-installed on modern Colab runtimes.
# torch-geometric version pin dropped — 2.4.0 has no wheel for Python 3.12+.
# pandas/networkx version pins dropped — pinned versions predate Python 3.12.
# rdkit-pypi renamed to rdkit.
import torch, subprocess

TORCH_VERSION = torch.__version__.split('+')[0]
result = subprocess.run(['nvcc', '--version'], capture_output=True, text=True)
cuda_str = result.stdout.split('release ')[-1].split(',')[0].strip().replace('.', '')
CUDA_TAG = f'cu{cuda_str}'
print(f'Torch: {TORCH_VERSION}, CUDA: {CUDA_TAG}')

# PyG sparse extensions (likely pre-installed; no-op if already present)
!pip install -q torch-scatter torch-sparse torch-cluster torch-spline-conv \
    -f https://data.pyg.org/whl/torch-{TORCH_VERSION}+{CUDA_TAG}.html

!pip install -q torch-geometric
!pip install -q networkx pandas rdkit pyemd

In [ ]:
# Cell 5: Smoke-test imports
import torch_geometric
import networkx, pandas, numpy
print('PyTorch:   ', torch.__version__)
print('PyG:       ', torch_geometric.__version__)
print('NetworkX:  ', networkx.__version__)
print('pandas:    ', pandas.__version__)
print('numpy:     ', numpy.__version__)
print('All imports OK')

In [ ]:
# Cell 6: Verify trained GNN checkpoint exists
import os
ckpt = 'param/gnns/mutag_gcn.pt'
assert os.path.exists(ckpt), f'Missing checkpoint: {ckpt} — was it included in the zip?'
print('Checkpoint found:', ckpt)

## Deviations from the original D4Explainer paper

All changes below are memory/runtime mitigations introduced to make training feasible on a single Colab A100 (40 GB). Each row flags whether the paper's math is preserved exactly, approximately, or changed.

| Change | Flag | Fidelity | Notes |
|---|---|---|---|
| **Per-sigma BCE loss** | always on when size_bucketed/gradient_checkpointing used | **Exact** | Original: one forward over all 10 sigmas, one backward. New: 10 forward+backward, each BCE scaled by 1/sigma_length. Sum of gradients is algebraically identical to the original. `total_sigmas` kwarg on `loss_func_bce` preserves the `1/|Σ|` weight. |
| **CF loss: single-sigma approximation** | always on when per-sigma BCE used | **Approximation** | Original: CF loss uses the mean of scores across all 10 sigmas as soft edge weights into the frozen GNN. New: CF loss uses only the last sigma's score. Same expected direction but higher-variance gradient estimate. May lower reported fidelity slightly vs paper numbers. TODO: implement detached-mean-with-grad-on-one-sigma if the baseline diverges from paper metrics. |
| **Gradient checkpointing on Powerful** | `--gradient_checkpointing` | **Exact (math); ~2× compute** | Recomputes the full `Powerful` forward during backward. No numerical change, just compute/memory tradeoff. Off by default — only enable when OOM can't be solved by bucketing. |
| **Mixed-precision (fp16) training** | `--use_amp` | **Approximation (numerical)** | ~2× memory reduction on activations. Small numerical drift vs fp32 baseline, usually harmless but should be off for the paper-comparison run. |
| **Reduced batch size** | `--train_batchsize` | **Exact (loss); different SGD noise** | Changing bsz changes SGD variance and effective LR, so it's a training-dynamics change even though the per-sample loss is identical. Paper likely uses bsz=32 but doesn't specify all datasets. |
| **Size-bucketed batch sampler** | `--size_bucketed` | **Exact (loss); different SGD sampling** | Groups graphs by node count so each batch pads to a tight N. Changes batch composition (batches are no longer i.i.d. samples) but loss per sample is unchanged. Required for Mutagenicity: naive random batching produces outlier batches with padded N≈197 that OOM even on A100 40 GB. |
| **Max-graph-size filter** | `--max_graph_size` | **Data change** | Drops training graphs above a node-count threshold (~0.3% of Mutagenicity at N>100). This is a *training-set reduction*, not a loss change. Document the kept-fraction when reporting results. |
| **--debug_shapes** | `--debug_shapes` | N/A (diagnostic only) | Prints padded N and CUDA memory for the first 5 batches. No training effect. |

### Dataset naming caveat
The repo's `--dataset mutag` flag loads **Mutagenicity** (4337 graphs, max ~417 nodes), *not* classic TU MUTAG (188 graphs, max 28). The D4Explainer authors preserved this imprecise label; both the code and the paper tables refer to "MUTAG" but train on Mutagenicity.

### Recommended baseline-vs-extension protocol
For the natural-gradient extension comparison, run baseline with:
- `--size_bucketed --max_graph_size 100` (required for Mutagenicity to avoid OOM)
- No `--use_amp`, no `--gradient_checkpointing` (minimize numerical drift)
- Default `--train_batchsize 32`

Then run the natural-gradient extension with the *same* flags. This isolates the extension's contribution from the memory-management changes.

---

### Extension (NOT paper — novel contribution)

| Change | Flag | Fidelity | Notes |
|---|---|---|---|
| **Natural-gradient hook on edge probabilities** | `--natural_gradient` | **Extension** | Registers a backward hook on each `score_batch` tensor that rescales `dL/d(score)` by the diagonal Fisher-Rao inverse metric `I(θ)^{-1} = θ(1-θ)`. Per CONTEXT.md §6 Alg 2: each score tensor is a point on the Bernoulli-product manifold `M_G`, so the natural-gradient correction is element-wise — O(|E|), not O(|E|²). Applied to all forward passes (BCE per-sigma + CF). No change to loss, architecture, or data. |
| **Natural-gradient boundary clamp** | `--nat_grad_eps` (default 1e-6) | **Extension hyperparameter** | θ is clamped to `[eps, 1-eps]` inside the hook to avoid vanishing/exploding gradient at the manifold boundary. Rule-of-thumb default; tune if saturation is observed. |

**Experimental protocol:**
- Baseline run: `!python main.py --dataset mutag --size_bucketed --max_graph_size 100`
- Extension run: **same flags + `--natural_gradient`** — single-flag toggle keeps the comparison clean
- With flag off, the code path is byte-for-byte identical to baseline


In [ ]:
# Cell 7: Sanity-check training (1 epoch)
# --verbose 1: run evaluation and save best_model.pth every epoch (needed when --epoch 1)
# --size_bucketed: group graphs by node count into tight-padding batches (see deviations cell below)
# --max_graph_size 100: drop ~0.3% outlier training graphs (>100 nodes) for Mutagenicity
# NOTE: repo flag `--dataset mutag` loads Mutagenicity (4337 graphs, max ~417 nodes), NOT classic TU MUTAG (188 graphs, max 28).
!python main.py --dataset mutag --epoch 1 --verbose 1 --size_bucketed --max_graph_size 100

In [ ]:
# Cell 8: Full training runs — memory flags depend on dataset graph size
#
# Mutagenicity (called `mutag` in repo, max N~417, highly size-heterogeneous):
#   !python main.py --dataset mutag --size_bucketed --max_graph_size 100
#
# ba3 (BA3Motif, moderate size): default loader is fine
#   !python main.py --dataset ba3
#
# Node-classification syn datasets (one big graph each): default loader is fine
#   !python main.py --dataset Tree_Cycle
#   !python main.py --dataset BA_shapes
#   !python main.py --dataset Tree_Grids
#
# Large-graph datasets (NCI1, bbbp): size-bucket + reduced bsz; add memory flags only if still OOM
#   !python main.py --dataset NCI1 --size_bucketed --train_batchsize 16
#   !python main.py --dataset bbbp --size_bucketed --train_batchsize 16
#   # If OOM persists, layer on: --gradient_checkpointing --use_amp

!python main.py --dataset mutag --size_bucketed --max_graph_size 100

In [ ]:
# Cell 9: Extension run — natural gradient on edge-probability manifold
# Uses SAME memory flags as Cell 8 baseline so the comparison isolates the extension's effect.
# Flag off = paper baseline (byte-identical code path); flag on = natural-gradient hook.
!python main.py --dataset mutag --size_bucketed --max_graph_size 100 --natural_gradient